In [1]:
# Calculate the PM2.5 Mortality per health variable

In [2]:
import os
import xarray as xr
import numpy as np

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
RR_DIR = "/glade/work/awells/air_quality/CESM/pm25/RR/"

In [9]:
# === Health variables ===
# COPD (chronic obstructive pulonary disease)
# LRI (lower respiratory infection)
# IHD (ischemic heart disease)
# DM2 (type 2 diabetes)
# LC (tracheal, bronchus, and lung cancer)
# Stroke
health_vars = ["DM", "LC", "COPD", "LRI"]
age_health_vars = ["Stroke", "IHD"]

In [13]:
# === Main loop ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/mortality/pm25/"
SCENARIOS = ["ARISE", "SSP245"]

for health_VAR in health_vars:
    for scenario in SCENARIOS:
        for ens_num in range(1, 11):
            print(f"Processing {scenario} ensemble member {ens_num:02d} for {health_VAR}")
            # Load data arrays
            if scenario == "ARISE":
                dates = "2035-2069"
            elif scenario == "SSP245":
                dates = "2020-2069"

            RR_file = f"RR_{health_VAR}_{scenario}_{ens_num:02d}_{dates}.nc"
            RR_path = os.path.join(RR_DIR, RR_file)
            RR = xr.open_dataarray(RR_path)

            PAF = 1 - (1/RR)

            bmr_file = f"GBD_BMR_Country_{health_VAR}_newlabels_1990-2009.nc"
            bmr_path = os.path.join(BMR_DIR, bmr_file)
            bmr = xr.open_dataarray(bmr_path)
            
            # Extract mean, lower, upper
            BMR_mean = bmr.sel(quantile="mean")
            BMR_lower = bmr.sel(quantile="lower")
            BMR_upper = bmr.sel(quantile="upper")
            
            # Estimate standard deviation assuming 95% CI
            BMR_std = (BMR_upper - BMR_lower) / (2 * 1.96)
            
            # Monte Carlo sampling (shape: country:204, sample:1000)
            BMR_samples = np.random.normal(
                loc=BMR_mean.values[..., np.newaxis],
                scale=BMR_std.values[..., np.newaxis],
                size=(len(bmr.country), 1000)
            )
            
            # Convert to xarray
            BMR_samples = xr.DataArray(
                BMR_samples,
                dims=("country", "sample"),
                coords={
                    "country": bmr.country,
                    "sample": np.arange(1000)
                }
            )
            
            mortality = BMR_samples * PAF

            description = (f"PM2.5 Attributable Mortality for {health_VAR} "
                           "- scripts by A.F. Wells (2025)")

            mortality.attrs["description"] = description
            mortality.attrs["ensemble"] = ens_num
            mortality.attrs["scenario"] = scenario

            out_file = f"PM2.5_Mortality_{health_VAR}_CESM_{scenario}_{ens_num:02d}_{dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)
            mortality.to_netcdf(out_path)


Processing ARISE ensemble member 01 for DM
Processing ARISE ensemble member 02 for DM
Processing ARISE ensemble member 03 for DM
Processing ARISE ensemble member 04 for DM
Processing ARISE ensemble member 05 for DM
Processing ARISE ensemble member 06 for DM
Processing ARISE ensemble member 07 for DM
Processing ARISE ensemble member 08 for DM
Processing ARISE ensemble member 09 for DM
Processing ARISE ensemble member 10 for DM
Processing SSP245 ensemble member 01 for DM
Processing SSP245 ensemble member 02 for DM
Processing SSP245 ensemble member 03 for DM
Processing SSP245 ensemble member 04 for DM
Processing SSP245 ensemble member 05 for DM
Processing SSP245 ensemble member 06 for DM
Processing SSP245 ensemble member 07 for DM
Processing SSP245 ensemble member 08 for DM
Processing SSP245 ensemble member 09 for DM
Processing SSP245 ensemble member 10 for DM
Processing ARISE ensemble member 01 for LC
Processing ARISE ensemble member 02 for LC
Processing ARISE ensemble member 03 for LC
P

In [16]:
mortality

<xarray.DataArray (country: 204, sample: 1000, year: 50)> Size: 82MB
array([[[ 0.34089737,  0.7662632 ,  1.3204836 , ...,  1.17264441,
          1.32401606,  0.88132541],
        [ 0.24115809,  0.19716641,  2.28390192, ...,  0.7712776 ,
          0.7394357 ,  0.01129856],
        [ 0.75086924,  0.71317405,  1.23542492, ...,  0.69558833,
          0.45806476,  0.38939719],
        ...,
        [ 0.52154095,  0.70515145,  1.58697012, ...,  0.62964922,
          1.38620045,  1.39098376],
        [ 0.44932212,  0.6832257 ,  1.55083451, ...,  0.59083243,
          0.70947231,  1.11171974],
        [ 0.66275938,  0.68849562,  1.47102134, ...,  1.07196953,
          1.17895525,  1.0979294 ]],

       [[ 2.20662762,  2.41438019,  4.01484069, ...,  5.14322404,
          2.07362388,  3.20315148],
        [ 2.78076986,  3.84492071,  2.52442103, ...,  3.63395508,
          3.61686926,  4.33689145],
        [ 2.9641051 ,  2.45015104,  2.64223164, ...,  3.79009432,
          4.13496427,  3.66646743],
...
        [ 1.27400823,  0.02347091,  0.57068669, ...,  1.43753335,
          1.35793771,  1.7102419 ],
        [ 0.98509275,  1.59956696,  1.02498068, ...,  1.11052981,
          0.94640813,  1.10134719],
        [ 1.5894791 ,  1.05378186,  1.04500967, ...,  1.93378283,
          1.58911997,  1.10949235]],

       [[ 0.18875636,  0.07314898, -0.15611746, ...,  0.3027553 ,
          0.29292968,  0.3328535 ],
        [-0.06343338,  0.18505239,  0.72337417, ...,  0.26250942,
          0.1095329 ,  0.3112508 ],
        [ 0.37868748,  0.49394072,  0.14665578, ...,  0.11668885,
          0.61764472,  0.30716109],
        ...,
        [ 0.45746755,  0.45386204,  0.43822673, ...,  0.46808415,
          0.54778815,  0.10663135],
        [-0.16331618,  0.2878356 ,  0.34933335, ...,  0.21860639,
          0.04378778,  0.42940726],
        [ 0.21589494,  0.04758622,  0.14589572, ...,  0.10130937,
          0.36038945,  0.23952803]]])
Coordinates:
  * country  (country) <U32 26kB 'American Samoa' ... 'United States'
  * sample   (sample) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999
  * year     (year) int64 400B 2020 2021 2022 2023 2024 ... 2066 2067 2068 2069
Attributes:
    descriptiopn:  PM2.5 Attributable Mortality for LRI - scripts by A.F. Wel...
    ensemble:      10
    scenario:      SSP245